In [1]:
from pathlib import Path
import pandas as pd

from decent_bench.utils.checkpoint_manager import CheckpointManager

In [2]:
metrics = {"accuracy", "loss", "consensus error", "nr gradient calls"}

In [3]:
checkpoint_dirs = Path(r"/home/ubuntu/github/decent-bench/examples/nim/results/heterogeneous")
if not checkpoint_dirs.is_dir():
    raise ValueError(f"Checkpoint directory {checkpoint_dirs} does not exist")

In [4]:
data = []
for checkpoint_dir in checkpoint_dirs.iterdir():
    for checkpoint in checkpoint_dir.iterdir():
        if not checkpoint.is_dir():
            display(f"Skipping {checkpoint} since it is not a directory")
            continue
        checkpoint_manager = CheckpointManager(checkpoint)
        try:
            metrics_result = checkpoint_manager.load_metrics_result(skip_agent_metrics=True)
        except Exception as e:
            display(f"Error loading metrics result for {checkpoint}: {e}")
            continue
        for alg, table in metrics_result.table_results.items():
            alg_data = {"algorithm": alg.name + checkpoint.name.replace("mnist", "")}
            for metric, values in table.items():
                if metric.table_description not in metrics:
                    continue
                for statistic, value in values.items():
                    if statistic != "avg":
                        continue
                    alg_data[metric.table_description] = value[0]
            data.append(alg_data)
df = pd.DataFrame(data)

In [6]:
df

,algorithm,consensus error,nr gradient calls,accuracy,loss
0,ProxSkip-ss-0.001-aux_ss-0.001-comm_prob-0.1-c...,0.013065,64000.0,0.17138,2.235432
1,ProxSkip-ss-0.001-aux_ss-0.001-comm_prob-0.1-c...,0.014711,64000.0,0.17124,2.226421
2,ProxSkip-ss-0.001-aux_ss-0.001-comm_prob-0.1-c...,0.030747,64000.0,0.16972,2.200981
3,ProxSkip-ss-0.001-aux_ss-0.001-comm_prob-0.5-c...,0.001458,64000.0,0.17246,2.255569
4,ProxSkip-ss-0.001-aux_ss-0.001-comm_prob-0.5-c...,0.003677,64000.0,0.17246,2.251918
...,...,...,...,...,...
723,LT-ADMM-EMA-OPT-ls-15-ss1-0.01-ss2-0.01-p-1.0-...,2.683884,480000.0,0.20684,1.065918
724,LT-ADMM-EMA-OPT-ls-15-ss1-0.01-ss2-0.01-p-1.0-...,0.618131,480000.0,0.54540,0.774720
725,LT-ADMM-EMA-OPT-ls-15-ss1-0.01-ss2-0.01-p-1.5-...,0.004178,480000.0,0.10500,2.364332
726,LT-ADMM-EMA-OPT-ls-15-ss1-0.01-ss2-0.01-p-1.5-...,1.333648,480000.0,0.37224,35.572376
